# 05 · Weather-adjusted growth, what drives the model, and its biggest misses

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
con = duckdb.connect("data/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")
q = lambda sql: con.execute(sql).df()
import statsmodels.formula.api as smf
import plotly.graph_objects as go
from viz import add_caption, save, SERIES, INK, INK_2

## 1. Weather-adjusted demand growth
Peak load rose about 15% from 2019 to the 2024 record. Is that growth, or hotter summers?
Fit **log(load)** on temperature (cooling and heating degrees and their squares, each allowed to differ by hour of day),
hour, weekday, month and holidays, plus a **year effect**. The year effect is the change in load at identical weather and calendar.
Uri (Feb 10–20 2021) and the Hurricane Beryl outage period (Jul 8–17 2024) are excluded because load there was lost to outages, not weather.
A second fit uses only summer afternoons (Jun–Sep, hour ending 2–8 pm), the hours that set the annual peak.

In [2]:
df = q("SELECT * FROM model_hourly WHERE year BETWEEN 2019 AND 2025 ORDER BY period")
w = df[~df.local_date.between("2021-02-10", "2021-02-20")
       & ~df.local_date.between("2024-07-08", "2024-07-17")
       & ~df.is_imputed.astype(bool)].copy()
w["cdd"] = (w.avg_temp_c - 18.3).clip(lower=0)
w["hdd"] = (18.3 - w.avg_temp_c).clip(lower=0)
w["log_d"] = np.log(w.demand_mwh)
YEARS = range(2020, 2026)

all_hours = smf.ols("log_d ~ C(year) + (cdd + I(cdd**2) + hdd + I(hdd**2)):C(local_hour)"
                    " + C(local_hour) + C(day_of_week) + is_holiday + C(month)", w).fit()
summer = w[w.month.between(6, 9) & w.local_hour.between(14, 20)]
summer_fit = smf.ols("log_d ~ C(year) + (cdd + I(cdd**2)):C(local_hour) + C(local_hour)"
                     " + C(day_of_week) + is_holiday + C(month)", summer).fit()
print(f"R^2 all hours {all_hours.rsquared:.3f}, summer afternoons {summer_fit.rsquared:.3f}")

def year_pct(m):
    return {2019: 0.0, **{y: 100 * (np.exp(m.params[f"C(year)[T.{y}]"]) - 1) for y in YEARS}}
def year_ci(m):
    return {2019: 0.0, **{y: 100 * 1.96 * m.bse[f"C(year)[T.{y}]"] for y in YEARS}}

raw_avg = w.groupby("year").demand_mwh.mean()
peak = q("SELECT year, peak_mwh FROM peak_growth_yearly").set_index("year").peak_mwh
growth = pd.DataFrame({
    "raw average load vs 2019 (%)": 100 * (raw_avg / raw_avg[2019] - 1),
    "weather-adjusted, all hours (%)": year_pct(all_hours),
    "weather-adjusted, summer afternoons (%)": year_pct(summer_fit),
    "raw annual peak vs 2019 (%)": 100 * (peak / peak[2019] - 1),
    "summer cooling degrees (avg °C above 18.3)": w[w.month.between(6, 9)].groupby("year").cdd.mean(),
}).round(2)
growth.to_csv("data/clean/weather_adjusted_growth.csv")
growth

/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_30152/685958041.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  q = lambda sql: con.execute(sql).df()


R^2 all hours 0.956, summer afternoons 0.931


,raw average load vs 2019 (%),"weather-adjusted, all hours (%)","weather-adjusted, summer afternoons (%)",raw annual peak vs 2019 (%),summer cooling degrees (avg °C above 18.3)
2019,0.00,0.00,0.00,0.00,10.11
2020,-0.96,0.33,-1.39,-0.49,9.53
2021,1.83,4.35,4.73,-1.42,8.87
2022,12.24,9.33,6.36,7.11,10.67
2023,16.46,14.92,11.70,14.62,11.72
2024,20.20,20.73,15.15,14.77,9.85
2025,27.22,26.00,18.39,12.16,9.80


In [3]:
# Where the growth is: average load by ERCOT weather zone, full clean years only
# (zone data starts May 2019 and drifts from the ERCOT total in 2024-25, see qa_zone_mismatch)
q("""SELECT zone,
            ROUND(AVG(demand_mwh) FILTER (WHERE year(period) = 2020)) AS avg_2020_mwh,
            ROUND(AVG(demand_mwh) FILTER (WHERE year(period) = 2023)) AS avg_2023_mwh,
            ROUND(100 * (AVG(demand_mwh) FILTER (WHERE year(period) = 2023)
                       / AVG(demand_mwh) FILTER (WHERE year(period) = 2020) - 1), 1) AS growth_pct
     FROM fact_zone_load GROUP BY zone ORDER BY growth_pct DESC""")

,zone,avg_2020_mwh,avg_2023_mwh,growth_pct
0,FWES,3682.0,5832.0,58.4
1,NRTH,943.0,1373.0,45.6
2,SCEN,6906.0,8419.0,21.9
3,SOUT,3478.0,4040.0,16.2
4,EAST,1625.0,1822.0,12.1
5,COAS,12210.0,13551.0,11.0
6,NCEN,13153.0,14519.0,10.4
7,WEST,1375.0,1449.0,5.4


In [4]:
fig = go.Figure()
series = [("raw average load vs 2019 (%)", "Actual average load", "Actual"),
          ("weather-adjusted, all hours (%)", "Weather-adjusted, all hours", "Weather-adjusted"),
          ("weather-adjusted, summer afternoons (%)", "Weather-adjusted, summer afternoons", "Summer afternoons")]
for i, (col, name, short) in enumerate(series):
    fig.add_scatter(x=growth.index, y=growth[col], mode="lines+markers", name=name,
                    line=dict(color=SERIES[i], width=2.5), marker=dict(size=8),
                    hovertemplate=f"{short} · %{{x}}: %{{y:+.1f}}%<extra></extra>")
    fig.add_annotation(x=2025, y=growth[col].iloc[-1], text=f"{short} {growth[col].iloc[-1]:+.0f}%", showarrow=False,
                       xanchor="left", xshift=8, font=dict(color=INK))
fig.add_hline(y=0, line=dict(color=INK_2, width=1))
fig.update_layout(title="ERCOT load growth since 2019, with and without weather",
                  xaxis_title="Year", yaxis_title="Change vs 2019 (%)", xaxis=dict(tickvals=list(range(2019, 2026)), range=[2018.7, 2026.3]),
                  margin=dict(r=190))
adj = growth["weather-adjusted, all hours (%)"]
add_caption(fig, f"At identical weather, ERCOT load grew {adj[2025]:.0f}% from 2019 to 2025, about {100*((1+adj[2025]/100)**(1/6)-1):.1f}% a year. Hotter summers are not the explanation.")
save(fig, "weather_adjusted_growth")

## 2. What drives the model: each city separately
Same day-ahead setup as notebook 02 (train 2019–2023, predict the change from yesterday, test 2024–2025), but with each of the 8 cities' temperature (and its value 24 h earlier) as separate inputs instead of the weighted average.
**Grouped permutation importance:** shuffle one group of inputs on the test set (for a city, both its temperature columns together), and measure how much MAPE rises. Five shuffles per group.
Caveat: city temperatures are correlated, so cities that move together split the credit. A low score means "adds little the other cities don't already carry", not "irrelevant"

In [5]:
from sklearn.ensemble import HistGradientBoostingRegressor
cities = q("PIVOT (SELECT period, city, temp_c FROM fact_weather) ON city USING first(temp_c)")
name_of = {f"t_{c.lower().replace(' ', '_')}": c for c in cities.columns[1:]}
cities.columns = ["period"] + list(name_of)
d = df.merge(cities, on="period").sort_values("period")
tcols = list(name_of)
for c in tcols:
    d[c + "_lag24"] = d[c].shift(24)
d["is_weekend"] = d.is_weekend.astype(int)
d["is_holiday"] = d.is_holiday.astype(int)
d = d.dropna(subset=["demand_lag_168"] + [c + "_lag24" for c in tcols])
base = ["demand_lag_24", "demand_lag_168", "local_hour", "day_of_week", "month", "is_weekend", "is_holiday"]
feats = base + tcols + [c + "_lag24" for c in tcols]
tr, te = d[d.period.dt.year <= 2023], d[d.period.dt.year >= 2024]
gb = HistGradientBoostingRegressor(random_state=0).fit(tr[feats], tr.demand_mwh - tr.demand_lag_24)
def mape_of(X):
    p = te.demand_lag_24.values + gb.predict(X[feats])   # anchor stays the true value; only model inputs are shuffled
    return 100 * np.mean(np.abs(te.demand_mwh.values - p) / te.demand_mwh.values)
base_mape = mape_of(te)
print(f"per-city model, test MAPE: {base_mape:.2f}%  (weighted-average model in notebook 02: 2.94%)")

groups = {name_of[c]: [c, c + "_lag24"] for c in tcols}
groups.update({"Demand 24 h earlier": ["demand_lag_24"], "Demand 1 week earlier": ["demand_lag_168"],
               "Hour of day": ["local_hour"], "Day of week": ["day_of_week"], "Month": ["month"],
               "Holiday / weekend": ["is_holiday", "is_weekend"]})
rng = np.random.default_rng(0)
rows = []
for g, cols in groups.items():
    rises = []
    for _ in range(5):
        X = te.copy()
        idx = rng.permutation(len(X))
        for c in cols:
            X[c] = X[c].values[idx]
        rises.append(mape_of(X) - base_mape)
    rows.append({"input": g, "is_city": g in name_of.values(), "mape_increase_pts": np.mean(rises), "sd": np.std(rises)})
imp = pd.DataFrame(rows).sort_values("mape_increase_pts", ascending=False).round(3)
imp.to_csv("data/clean/feature_importance.csv", index=False)
imp

/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_30152/685958041.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  q = lambda sql: con.execute(sql).df()


per-city model, test MAPE: 2.94%  (weighted-average model in notebook 02: 2.94%)


,input,is_city,mape_increase_pts,sd
8,Demand 24 h earlier,False,2.336,0.016
2,Dallas,True,0.886,0.007
0,Austin,True,0.643,0.006
11,Day of week,False,0.615,0.017
10,Hour of day,False,0.346,0.002
1,Corpus Christi,True,0.307,0.004
3,Houston,True,0.307,0.008
6,San Antonio,True,0.229,0.004
7,Tyler,True,0.163,0.006
5,Midland,True,0.102,0.004


In [6]:
srt = imp.sort_values("mape_increase_pts")
fig = go.Figure()
for is_city, label, color in [(True, "City temperature", SERIES[0]), (False, "Other input", SERIES[1])]:
    s = srt[srt.is_city == is_city]
    fig.add_bar(y=s.input, x=s.mape_increase_pts, orientation="h", name=label, marker_color=color,
                error_x=dict(type="data", array=s.sd, color=INK_2, thickness=1),
                hovertemplate="%{y}: +%{x:.2f} pts<extra></extra>")
fig.update_layout(title="What drives the day-ahead forecast (test set 2024–2025)", barmode="overlay", bargap=0.3,
                  xaxis_title="Rise in MAPE when the input is shuffled (percentage points)", yaxis_title="",
                  yaxis=dict(categoryorder="array", categoryarray=list(srt.input)), margin=dict(l=170))
top_city = imp[imp.is_city].iloc[0]
add_caption(fig, f"Yesterday's load matters most; among cities, {top_city.input} leads. The four added cities together matter less than Dallas alone.")
save(fig, "feature_importance", height=680)

## 3. The model's 10 biggest misses
Worst days on the test set by daily MAPE for the main model (notebook 02). The context column is generated from the data: holidays, the day after a holiday (the model anchors on yesterday's load), freezing or extreme heat, and big temperature swings from the day before. The one hand-labelled event is Hurricane Beryl.

In [7]:
t = pd.read_parquet("data/clean/test_predictions.parquet")
t["local_date"] = t.local_ts.dt.normalize()
t["err"] = t.pred_best - t.demand_mwh
hol = pd.read_parquet("data/clean/us_holidays.parquet")
hol = dict(zip(pd.to_datetime(hol.holiday_date), hol.holiday_name))
tl = q("SELECT period, avg_temp_c - temp_lag_24 AS temp_delta_24 FROM model_hourly")
t = t.merge(tl, on="period", how="left")
KNOWN = {pd.Timestamp("2024-07-08"): "Hurricane Beryl made landfall near Houston; about 2.2 million customers lost power"}

def worst_hour(g):
    r = g.loc[g.err.abs().idxmax()]
    return pd.Series({"worst_hour_ending": int(r.local_hour), "worst_hour_error_gw": r.err / 1000})

days = t.groupby("local_date").agg(our_mape=("ape_best", "mean"), ercot_mape=("ape_ercot", "mean"),
                                   mean_error_gw=("err", lambda e: e.mean() / 1000), min_temp=("avg_temp_c", "min"),
                                   max_temp=("avg_temp_c", "max"), max_swing=("temp_delta_24", lambda s: s.abs().max()))
days = days.join(t.groupby("local_date")[["err", "local_hour"]].apply(worst_hour))
worst = days.sort_values("our_mape", ascending=False).head(10).copy()

def context(day, r):
    notes = []
    if day in KNOWN: notes.append(KNOWN[day])
    if day in hol: notes.append(f"Holiday: {hol[day]}")
    y = day - pd.Timedelta(days=1)
    if y in hol: notes.append(f"Day after {hol[y]}: the model anchors on yesterday's holiday-level load")
    if day - pd.Timedelta(days=2) in hol and y not in hol:
        notes.append(f"Two days after {hol[day - pd.Timedelta(days=2)]}")
    if r.min_temp <= 0: notes.append(f"Freezing: low of {r.min_temp:.0f} °C")
    if r.max_temp >= 37: notes.append(f"Extreme heat: high of {r.max_temp:.0f} °C")
    if r.max_swing >= 8: notes.append(f"Temperature swung up to {r.max_swing:.0f} °C from the day before")
    return "; ".join(notes) or "No clear driver in our data"

worst["weekday"] = worst.index.day_name()
worst["direction"] = np.where(worst.mean_error_gw > 0, "over-forecast", "under-forecast")
worst["context"] = [context(day, r) for day, r in worst.iterrows()]
worst = worst.reset_index()[["local_date", "weekday", "our_mape", "ercot_mape", "direction", "worst_hour_ending",
                             "worst_hour_error_gw", "min_temp", "max_temp", "max_swing", "context"]].round(1)
worst["local_date"] = worst.local_date.dt.date
worst.to_csv("data/clean/biggest_misses.csv", index=False)
worst

/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_30152/685958041.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  q = lambda sql: con.execute(sql).df()
/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_30152/3250889067.py:37: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  "worst_hour_error_gw", "min_temp", "max_temp", "max_swing", "context"]].round(1)


,local_date,weekday,our_mape,ercot_mape,direction,worst_hour_ending,worst_hour_error_gw,min_temp,max_temp,max_swing,context
0,2024-01-15,Monday,16.3,4.8,under-forecast,8.0,-23.0,-7.1,0.8,7.6,Holiday: Martin Luther King Jr. Day; Freezing:...
1,2024-07-08,Monday,12.8,15.4,over-forecast,17.0,12.6,24.6,28.9,4.0,Hurricane Beryl made landfall near Houston; ab...
2,2025-01-20,Monday,11.6,2.8,under-forecast,18.0,-12.1,-4.0,2.2,4.8,Holiday: Martin Luther King Jr. Day; Freezing:...
3,2024-01-16,Tuesday,10.9,4.6,under-forecast,5.0,-15.7,-8.9,-0.7,3.5,Day after Martin Luther King Jr. Day: the mode...
4,2025-03-08,Saturday,10.1,2.3,under-forecast,17.0,-9.9,10.8,21.0,7.6,No clear driver in our data
5,2025-01-02,Thursday,10.0,1.8,under-forecast,9.0,-6.7,6.2,16.3,4.0,Day after New Year's Day: the model anchors on...
6,2025-11-29,Saturday,9.1,1.0,under-forecast,17.0,-5.8,13.3,22.1,6.3,Two days after Thanksgiving Day
7,2024-01-14,Sunday,8.6,3.6,under-forecast,18.0,-12.0,-4.7,2.7,13.7,Freezing: low of -5 °C; Temperature swung up t...
8,2025-02-14,Friday,8.4,4.7,under-forecast,15.0,-7.3,3.8,12.7,7.0,No clear driver in our data
9,2025-11-11,Tuesday,8.4,5.7,under-forecast,9.0,-6.6,5.8,22.1,7.4,Holiday: Veterans Day
